# 3. Train Masking Model and Count Decoder



This step requires a GPU. Run on a GPU node (e.g., via your scheduler) or a machine with CUDA.

## 3.1. Train Masking Model (GPU required)


This step requires a GPU. Run on a GPU node (e.g., via your scheduler) or a machine with CUDA.


In [1]:
# GPU required
OUTPUT_DIR = "T_perturb/res/masking"  # relative to repo root, here set the output directory
SRC_DATASET = "T_perturb/tokenized_data/LPS_all_tps_2k/dataset_2000_hvg_src/normal.dataset" # path/to/src_dataset.dataset from tokenization step
TGT_DATASET_FOLDER = "T_perturb/tokenized_data/LPS_all_tps_2k/dataset_2000_hvg_tgt" # path/to/tgt_datasets_folder from tokenization step
SRC_ADATA = "T_perturb/tokenized_data/LPS_all_tps_2k/h5ad_pairing_2000_hvg_src/normal.h5ad" # path/to/src_adata.h5ad" from tokenization step
TGT_ADATA_FOLDER = "T_perturb/tokenized_data/LPS_all_tps_2k/h5ad_pairing_2000_hvg_tgt" # path/to/tgt_adata_folder from tokenization step
MAPPING_DICT_PATH = "T_perturb/tokenized_data/LPS_all_tps_2k/token_id_to_genename_2000_hvg.pkl" # path/to/mapping_dict.pkl from tokenization step

ENCODER_PATH = "/lustre/scratch126/cellgen/lotfollahi/av13/scmaskgit/foundation_107m/checkpoints/20250709_1223_cellgen_train_masking_lr_5e-05_wd_1e-06_batch_64_ptime_pos_sin_m_pow_tp_1-2-3_s_42-epoch=00.ckpt" # path to pretrained encoder checkpoint provided with Perturbgen

BATCH_SIZE = 64 # Model training batch size
MAX_LEN = 797 # maximum sequence length
EPOCHS = 5 # number of training epochs
TGT_VOCAB_SIZE = 2002 # target vocabulary size (number of tokens in target dataset)
CELLGEN_LR = 1e-4 # learning rate
CELLGEN_WD = 1e-4 # weight decay
N_WORKERS = 1 # number of data loading workers
NUM_LAYERS = 6 # number of transformer layers
D_FF = 32 # feedforward dimension
PRED_TPS = ["1", "2"] # time points to train on and predict (for LPS: "1"=90m, "2"=6h, "3"=10h)

VAR_LIST = ["cell_type_harmonized", "time_after_LPS"] # list of obs retained in adata.vars after preprocessing

SEED = 0 # random seed for reproducibility
CONTEXT_MODE = "True" # whether to use context tokens
POS_ENCODING_MODE = "time_pos_sin" # positional encoding mode
MASK_SCHEDULER = "pow" # masking scheduler type
NUM_NODE = 1 # number of nodes if using distributed training
D_MODEL = 768 # model dimension

CKPT_MASKING_PATH = None  # optional path to checkpoint to resume training from
USE_WEIGHTED_SAMPLER = "False" # whether to use weighted sampler during training

In [2]:
cmd = [
    "python",
    "-m",
    "perturbgen",
    "train-mask",
    "--train_mode", "masking",
    "--split", "False",
    "--encoder", "scmaskgit",
    "--splitting_mode", "stratified",
    "--split_obs", "cell_type_harmonized",
    "--output_dir", OUTPUT_DIR,
    "--src_dataset", SRC_DATASET,
    "--tgt_dataset_folder", TGT_DATASET_FOLDER,
    "--src_adata", SRC_ADATA,
    "--tgt_adata_folder", TGT_ADATA_FOLDER,
    "--mapping_dict_path", MAPPING_DICT_PATH,
    "--batch_size", str(BATCH_SIZE),
    "--epochs", str(EPOCHS),
    "--cellgen_lr", str(CELLGEN_LR),
    "--cellgen_wd", str(CELLGEN_WD),
    "--n_workers", str(N_WORKERS),
    "--num_layers", str(NUM_LAYERS),
    "--d_ff", str(D_FF),
    "--pred_tps", *PRED_TPS,
    "--var_list", *VAR_LIST,
    "--encoder_path", ENCODER_PATH,
    "--seed", str(SEED),
    "--context_mode", CONTEXT_MODE,
    "--pos_encoding_mode", POS_ENCODING_MODE,
    "--mask_scheduler", MASK_SCHEDULER,
    "--num_node", str(NUM_NODE),
    "--d_model", str(D_MODEL),
    "--use_weighted_sampler", USE_WEIGHTED_SAMPLER,
    "--ckpt_every_n_epochs", "1",
]

if CKPT_MASKING_PATH:
    cmd += ["--ckpt_masking_path", CKPT_MASKING_PATH]

print(" ".join(cmd))


python -m perturbgen train-mask --train_mode masking --split False --encoder scmaskgit --splitting_mode stratified --split_obs cell_type_harmonized --output_dir T_perturb/res/masking --src_dataset T_perturb/tokenized_data/LPS_all_tps_2k/dataset_2000_hvg_src/normal.dataset --tgt_dataset_folder T_perturb/tokenized_data/LPS_all_tps_2k/dataset_2000_hvg_tgt --src_adata T_perturb/tokenized_data/LPS_all_tps_2k/h5ad_pairing_2000_hvg_src/normal.h5ad --tgt_adata_folder T_perturb/tokenized_data/LPS_all_tps_2k/h5ad_pairing_2000_hvg_tgt --mapping_dict_path T_perturb/tokenized_data/LPS_all_tps_2k/token_id_to_genename_2000_hvg.pkl --batch_size 64 --epochs 5 --cellgen_lr 0.0001 --cellgen_wd 0.0001 --n_workers 1 --num_layers 6 --d_ff 32 --pred_tps 1 2 --var_list cell_type_harmonized time_after_LPS --encoder_path /lustre/scratch126/cellgen/lotfollahi/av13/scmaskgit/foundation_107m/checkpoints/20250709_1223_cellgen_train_masking_lr_5e-05_wd_1e-06_batch_64_ptime_pos_sin_m_pow_tp_1-2-3_s_42-epoch=00.ckpt -

In [3]:
import subprocess

subprocess.run(cmd, check=True)

loading, please wait...
Current working directory: /lustre/scratch126/cellgen/lotfollahi/kl11
Loading and preprocessing data...
Loading 2_10h_LPS.dataset...
Loading 1_90m_LPS.dataset...
Loading 2_10h_LPS.h5ad...
Loading 1_90m_LPS.h5ad...


Seed set to 42
/nfs/team361/cytomeister/.cytomeister/lib/python3.11/site-packages/anndata/_core/anndata.py:1758: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/nfs/team361/cytomeister/.cytomeister/lib/python3.11/site-packages/anndata/_core/anndata.py:1758: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


---PerturbGen training --- 
Target vocab size: 1518, max sequence length: 665
Using NVIDIA A100-SXM4-80GB for training
Set float32_matmul_precision to medium
-- Initializing scmaskgit model
Start datamodule
Using device gpu.


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
wandb: Currently logged in as: k-ly (mll_sut). Use `wandb login --relogin` to force relogin
wandb: Tracking run with wandb version 0.17.9
wandb: Run data is saved locally in logs/wandb/run-20260324_210111-8e8q0mmp
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run 20260324_2101_cellgen
wandb: ⭐️ View project at https://wandb.ai/mll_sut/ttransformer
wandb: 🚀 View run at https://wandb.ai/mll_sut/ttransformer/runs/8e8q0mmp
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name         | Type             | Params | Mode 
----------------------------------------------------------
0 | transformer  | PerturbGen       | 90.7 M | train
1 | masking_loss | CrossEntropyLoss | 0      | train
2 | perplexity   | Perplexity       | 0      | train
3 | mse          | MeanSquaredError | 0      | train
----------------------------------------------------------
19.0 M    Trainabl

Epoch 0: 100%|██████████| 1247/1247 [12:29<00:00,  1.66it/s, v_num=0mmp, lr=0.0001, train/perplexity_step=350.0, train/masking_loss_step=5.860, train/perplexity_epoch=165.0, train/masking_loss_epoch=5.000]

...

Epoch 4: 100%|██████████| 1247/1247 [12:28<00:00,  1.67it/s, v_num=0mmp, lr=0.0001, train/perplexity_step=247.0, train/masking_loss_step=5.510, train/perplexity_epoch=81.90, train/masking_loss_epoch=4.360]

Epoch 4, global step 6235: 'train/perplexity' reached 81.85079 (best 81.85079), saving model to '/lustre/scratch126/cellgen/lotfollahi/kl11/T_perturb/res/masking/checkpoints/20260324_2101_cellgen_train_masking_lr_0.0001_wd_0.0001_batch_64_ptime_pos_sin_m_pow_tp_1-2_s_0-epoch=04.ckpt' as top 5
`Trainer.fit` stopped: `max_epochs=5` reached.


Epoch 4: 100%|██████████| 1247/1247 [12:29<00:00,  1.66it/s, v_num=0mmp, lr=0.0001, train/perplexity_step=247.0, train/masking_loss_step=5.510, train/perplexity_epoch=81.90, train/masking_loss_epoch=4.360]


wandb: / 0.033 MB of 0.033 MB uploaded
wandb: Run history:
wandb:                    epoch ▁▁▁▁▁▁▁▁▃▃▃▃▃▃▃▃▅▅▅▅▅▅▅▅▆▆▆▆▆▆▆▆████████
wandb:                       lr ▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb: train/masking_loss_epoch █▅▃▂▁
wandb:  train/masking_loss_step █▆▆▅▆▅▆▄▅▅▅▄▄█▆▃▅▄▄▄▃▆▄▃▄▃▃▅▃▅▄▄▄▂▂▄▃▆▄▁
wandb:   train/perplexity_epoch █▄▂▁▁
wandb:    train/perplexity_step █▅▅▄▄▄▅▃▄▃▃▃▃█▄▂▃▂▃▃▂▅▃▂▃▂▂▃▂▄▃▃▂▂▂▃▂▄▃▁
wandb:      trainer/global_step ▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇███
wandb: 
wandb: Run summary:
wandb:                    epoch 4
wandb:                       lr 0.0001
wandb: train/masking_loss_epoch 4.35511
wandb:  train/masking_loss_step 3.84469
wandb:   train/perplexity_epoch 81.85079
wandb:    train/perplexity_step 46.74418
wandb:      trainer/global_step 6234
wandb: 
wandb: 🚀 View run 20260324_2101_cellgen at: https://wandb.ai/mll_sut/ttransformer/runs/8e8q0mmp
wandb: ⭐️ View project at: https://wandb.ai/mll_sut/ttransformer
wandb: Synced 5 W&B file(s), 0 

CompletedProcess(args=['python', '-m', 'perturbgen', 'train-mask', '--train_mode', 'masking', '--split', 'False', '--encoder', 'scmaskgit', '--splitting_mode', 'stratified', '--split_obs', 'cell_type_harmonized', '--output_dir', 'T_perturb/res/masking', '--src_dataset', 'T_perturb/tokenized_data/LPS_all_tps_2k/dataset_2000_hvg_src/normal.dataset', '--tgt_dataset_folder', 'T_perturb/tokenized_data/LPS_all_tps_2k/dataset_2000_hvg_tgt', '--src_adata', 'T_perturb/tokenized_data/LPS_all_tps_2k/h5ad_pairing_2000_hvg_src/normal.h5ad', '--tgt_adata_folder', 'T_perturb/tokenized_data/LPS_all_tps_2k/h5ad_pairing_2000_hvg_tgt', '--mapping_dict_path', 'T_perturb/tokenized_data/LPS_all_tps_2k/token_id_to_genename_2000_hvg.pkl', '--batch_size', '64', '--epochs', '5', '--cellgen_lr', '0.0001', '--cellgen_wd', '0.0001', '--n_workers', '1', '--num_layers', '6', '--d_ff', '32', '--pred_tps', '1', '2', '--var_list', 'cell_type_harmonized', 'time_after_LPS', '--encoder_path', '/lustre/scratch126/cellgen/l

## 3.2. Train count decoder (GPU required)

This step trains the count decoder using the masking checkpoint.


In [ ]:
# GPU required
COUNT_OUTPUT_DIR = "T_perturb/res/count"  # relative to repo root, here set the output directory
CKPT_MASKING_PATH = "/lustre/scratch126/cellgen/lotfollahi/kl11/T_perturb/res/masking/checkpoints/20260324_2101_cellgen_train_masking_lr_0.0001_wd_0.0001_batch_64_ptime_pos_sin_m_pow_tp_1-2_s_0-epoch=04.ckpt" # should be selected based on best masking model from previous step

SRC_DATASET = "T_perturb/tokenized_data/LPS_all_tps_2k/dataset_2000_hvg_src/normal.dataset" # path/to/src_dataset.dataset from tokenization step
TGT_DATASET_FOLDER = "T_perturb/tokenized_data/LPS_all_tps_2k/dataset_2000_hvg_tgt" # path/to/tgt_datasets_folder from tokenization step
SRC_ADATA = "T_perturb/tokenized_data/LPS_all_tps_2k/h5ad_pairing_2000_hvg_src/normal.h5ad" # path/to/src_adata.h5ad" from tokenization step
TGT_ADATA_FOLDER = "T_perturb/tokenized_data/LPS_all_tps_2k/h5ad_pairing_2000_hvg_tgt"  # path/to/tgt_adata_folder from tokenization step
MAPPING_DICT_PATH = "T_perturb/tokenized_data/LPS_all_tps_2k/token_id_to_genename_2000_hvg.pkl" # path/to/mapping_dict.pkl from tokenization step

ENCODER_PATH = "/lustre/scratch126/cellgen/lotfollahi/av13/scmaskgit/foundation_107m/checkpoints/20250709_1223_cellgen_train_masking_lr_5e-05_wd_1e-06_batch_64_ptime_pos_sin_m_pow_tp_1-2-3_s_42-epoch=00.ckpt" # path to pretrained encoder checkpoint provided with Perturbgen

BATCH_SIZE = 16 # Model training batch size
EPOCHS = 5 # number of training epochs
COUNT_LR = 0.001 # learning rate for count model
CELLGEN_LR = 0.0001 # learning rate for masking part, not useful here
CELLGEN_WD = 0.0001 # weight decay for masking part, not useful here
COUNT_WD = 0.001 # weight decay for count model
MLM_PROB = 0.30 # masking probability
N_WORKERS = 1 # number of data loading workers
NUM_LAYERS = 6 # number of transformer layers
D_FF = 32 # feedforward dimension
LOSS_MODE = "zinb" # loss mode for count model, could be mse, nb, or zinb
PRED_TPS = ["1", "2"] # time points to train on and predict (for LPS: "1"=90m, "2"=6h, "3"=10h)

VAR_LIST = ["cell_type_harmonized", "time_after_LPS"] # list of obs retained in adata.vars after preprocessing

COUNT_DROPOUT = 0.1 # dropout for count model
USE_POSITIONAL_ENCODING = "False" # whether to use positional encoding in count model
LAYER_NORM = "True" # whether to use layer normalization in count model
CONTEXT_MODE = "True" # whether to use context tokens
POS_ENCODING_MODE = "time_pos_sin" # positional encoding mode
MASK_SCHEDULER = "pow" # masking scheduler type
NUM_NODE = 1 # number of nodes if using distributed training
D_MODEL = 768 # model dimension


In [11]:
cmd = [
    "python",
    "-m",
    "perturbgen",
    "train-decoder",
    "--train_mode", "count",
    "--split", "False",
    "--splitting_mode", "stratified",
    "--output_dir", COUNT_OUTPUT_DIR,
    "--ckpt_masking_path", CKPT_MASKING_PATH,
    "--src_dataset", SRC_DATASET,
    "--tgt_dataset_folder", TGT_DATASET_FOLDER,
    "--src_adata", SRC_ADATA,
    "--tgt_adata_folder", TGT_ADATA_FOLDER,
    "--mapping_dict_path", MAPPING_DICT_PATH,
    "--batch_size", str(BATCH_SIZE),
    "--epochs", str(EPOCHS),
    "--count_lr", str(COUNT_LR),
    "--cellgen_lr", str(CELLGEN_LR),
    "--cellgen_wd", str(CELLGEN_WD),
    "--count_wd", str(COUNT_WD),
    "--mlm_prob", str(MLM_PROB),
    "--n_workers", str(N_WORKERS),
    "--num_layers", str(NUM_LAYERS),
    "--d_ff", str(D_FF),
    "--loss_mode", LOSS_MODE,
    "--pred_tps", *PRED_TPS,
    "--var_list", *VAR_LIST,
    "--encoder", "scmaskgit",
    "--count_dropout", str(COUNT_DROPOUT),
    "--use_positional_encoding", USE_POSITIONAL_ENCODING,
    "--layer_norm", LAYER_NORM,
    "--context_mode", CONTEXT_MODE,
    "--encoder_path", ENCODER_PATH,
    "--pos_encoding_mode", POS_ENCODING_MODE,
    "--mask_scheduler", MASK_SCHEDULER,
    "--num_node", str(NUM_NODE),
    "--d_model", str(D_MODEL),
    "--ckpt_every_n_epochs", "1",
]

print(" ".join(cmd))


python -m perturbgen train-decoder --train_mode count --split False --splitting_mode stratified --output_dir T_perturb/res/count --ckpt_masking_path /lustre/scratch126/cellgen/lotfollahi/kl11/T_perturb/res/masking/checkpoints/20260324_2101_cellgen_train_masking_lr_0.0001_wd_0.0001_batch_64_ptime_pos_sin_m_pow_tp_1-2_s_0-epoch=04.ckpt --src_dataset T_perturb/tokenized_data/LPS_all_tps_2k/dataset_2000_hvg_src/normal.dataset --tgt_dataset_folder T_perturb/tokenized_data/LPS_all_tps_2k/dataset_2000_hvg_tgt --src_adata T_perturb/tokenized_data/LPS_all_tps_2k/h5ad_pairing_2000_hvg_src/normal.h5ad --tgt_adata_folder T_perturb/tokenized_data/LPS_all_tps_2k/h5ad_pairing_2000_hvg_tgt --mapping_dict_path T_perturb/tokenized_data/LPS_all_tps_2k/token_id_to_genename_2000_hvg.pkl --batch_size 16 --epochs 16 --count_lr 0.001 --cellgen_lr 0.0001 --cellgen_wd 0.0001 --count_wd 0.001 --mlm_prob 0.3 --n_workers 1 --num_layers 6 --d_ff 32 --loss_mode zinb --pred_tps 1 2 --var_list cell_type_harmonized tim

In [12]:
import subprocess

subprocess.run(cmd, check=True)


loading, please wait...
Current working directory: /lustre/scratch126/cellgen/lotfollahi/kl11
Loading and preprocessing data...
Loading 2_10h_LPS.dataset...
Loading 1_90m_LPS.dataset...
Loading 2_10h_LPS.h5ad...
Loading 1_90m_LPS.h5ad...


Seed set to 42
/nfs/team361/cytomeister/.cytomeister/lib/python3.11/site-packages/anndata/_core/anndata.py:1758: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/nfs/team361/cytomeister/.cytomeister/lib/python3.11/site-packages/anndata/_core/anndata.py:1758: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


---PerturbGen training --- 
Target vocab size: 1518, max sequence length: 665
Using NVIDIA A100-SXM4-80GB for training
Set float32_matmul_precision to medium
-- Initializing scmaskgit model


/lustre/scratch126/cellgen/lotfollahi/kl11/T_perturb/perturbgen/Model/trainer.py:653: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(ckpt_masking_path

Start datamodule
Using device gpu.


...

Epoch 0:   0%|          | 0/4986 [00:00<?, ?it/s] 

/nfs/team361/cytomeister/.cytomeister/lib/python3.11/site-packages/pytorch_lightning/utilities/data.py:78: Trying to infer the `batch_size` from an ambiguous collection. The batch size we found is 16. To avoid any miscalculations, use `self.log(..., batch_size=batch_size)`.


Epoch 0:   8%|▊         | 410/4986 [01:20<15:01,  5.07it/s, v_num=vdeo, train/loss_step=603.0, train/mse_step=5.140] 


Aborted!

Detected KeyboardInterrupt, attempting graceful shutdown ...


KeyboardInterrupt: 